# Independent component analysis (ICA)

_Machine Learning_

**Unmix blended signals back into their separate sources.**

Imagine two people talking and two microphones. Each mic hears a mix of both voices.

     This is the cocktail party problem. We want the separate voices back.

---

This notebook builds the ICA unmixing demo up one step at a time. Run each cell, read the note above it, then tackle the practice problems at the bottom. _Save a copy to your Drive (File → Save a copy in Drive) to edit and keep your work._

In [ ]:
# Setup — numpy / pandas / scikit-learn / matplotlib ship with Colab.
import numpy as np
import matplotlib.pyplot as plt

## First, look at the data

Before training on it, see what each example actually contains. These are **images, not table columns** — each sample is an 8x8 grid of pixel intensities (0–16).

In [ ]:
from sklearn.datasets import load_digits

digits = load_digits()
print("image array:", digits.images.shape, " labels:", digits.target.shape)
samples = list(zip(digits.images, digits.target))
fig, axes = plt.subplots(1, 5, figsize=(8, 2))
for ax, (image, label) in zip(axes, samples):
    ax.imshow(image, cmap="gray")
    ax.set_title(str(label))
    ax.axis("off")
plt.show()

## Reference implementation — scikit-learn

We'll create two known source signals, **mix** them together, and then ask ICA to recover the originals — the cocktail-party problem in miniature. We build it in three steps: (1) make and mix the sources, (2) run FastICA to unmix, (3) check how well the recovered signals match the originals.

### Step 1 — Build two sources and mix them

We make two clearly *independent* signals: a smooth sine wave and a sharp square wave, plus a little noise. Stacking them gives the true source matrix `S`. The mixing matrix `A` then blends them, so each column of `X` is a different combination of both sources — exactly what two microphones would record.

In [ ]:
import numpy as np

rng = np.random.default_rng(0)
t = np.linspace(0, 8, 600)

# Two independent source signals.
s1 = np.sin(2 * t)              # source 1: smooth sine
s2 = np.sign(np.sin(3 * t))    # source 2: square wave
S = np.c_[s1, s2]
S += 0.1 * rng.normal(size=S.shape)   # a little observation noise

# Blend the two sources into two mixed channels.
A = np.array([[1.0, 0.7], [0.5, 1.0]])
X = S @ A.T
print("sources S:", S.shape, " mixtures X:", X.shape)

### Step 2 — Unmix with FastICA

ICA assumes the underlying sources are statistically **independent** and non-Gaussian, and it searches for an unmixing that makes the recovered channels as independent as possible. `fit_transform` returns the recovered sources `S_hat`, one column per component.

In [ ]:
from sklearn.decomposition import FastICA

ica = FastICA(n_components=2, random_state=0)
S_hat = ica.fit_transform(X)   # recovered sources
print("recovered shape:", S_hat.shape)

### Step 3 — Check the recovery

ICA can only recover sources **up to sign and order** — it might return them flipped or swapped. So instead of comparing directly, we correlate each recovered channel against each true source. A value near +1 or -1 in the correlation table means a strong match (the sign just tells us if it was flipped).

In [ ]:
# Correlate each recovered signal with each true source.
corr = np.corrcoef(S_hat.T, S.T)[:2, 2:]
print("recovered-vs-true correlations:")
print(np.round(corr, 2))

## Visualize the data & results

_From two blended digit-image intensity signals, can ICA recover the original independent sources?_

Now we run the same idea on real data: two average digit images become our sources, we mix them, and we plot the unmixed result. We build it in three steps.

### Step 1 — Build two sources from real digit images

We take every image of the digit 3 and average them into a single 64-pixel "mean 3" pattern, and do the same for the digit 8. Centring each (subtracting its mean) gives two real, structured source signals to unmix.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets import load_digits

digits = load_digits()
X, y = digits.data, digits.target

# Two real source signals: the mean pixel pattern of digits 3 and 8.
s1 = X[y == 3].mean(0)   # 64-pixel mean image of digit 3
s2 = X[y == 8].mean(0)   # 64-pixel mean image of digit 8
S = np.c_[s1, s2]
S = S - S.mean(0)        # centre each source
print("source matrix:", S.shape)

### Step 2 — Mix the sources, then unmix with ICA

We blend the two digit patterns with the same kind of mixing matrix as before, giving `Xmix`. FastICA then tries to undo the blend. We raise `max_iter` here because real, structured signals can take more iterations to converge.

In [ ]:
from sklearn.decomposition import FastICA

# Mix the two sources into two blended channels.
A = np.array([[1.0, 0.7], [0.5, 1.0]])
Xmix = S @ A.T

# Recover the independent sources.
S_hat = FastICA(n_components=2, random_state=0, max_iter=2000).fit_transform(Xmix)
print("recovered shape:", S_hat.shape)

### Step 3 — Plot the mixture against the recovered sources

We scale each signal to a common range (ICA fixes neither sign nor scale) and plot the mixture alongside the two recovered sources. If unmixing worked, the recovered curves should look like distinct, structured patterns rather than copies of the blended mixture.

In [ ]:
def scale(v):
    m = np.abs(v).max()
    return v / m if m > 0 else v

t = np.arange(64)
plt.plot(t, scale(Xmix[:, 0]), color="#ff7b72", label="mixture")
plt.plot(t, scale(S_hat[:, 0]), color="#4ea1ff", label="recovered source 1")
plt.plot(t, scale(S_hat[:, 1]), color="#7ee787", label="recovered source 2")
plt.xlabel("pixel index")
plt.ylabel("intensity (scaled)")
plt.title("ICA unmixing of digit-pixel signals")
plt.legend()
plt.show()